# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa – Exploration with `mlcroissant`
This notebook provides a practical guide for loading, exploring, and processing the FAIR² mental health survey dataset using the `mlcroissant` library.

### Dataset Source
This dataset is defined by a Croissant schema accessible at the provided URL and is AI-ready for rigorous analysis.

In [ ]:
# Ensure `mlcroissant` and plotting libraries are installed
!pip install mlcroissant matplotlib seaborn

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, their `@id`s, and accessible fields with their `@id`s.

In [ ]:
# List record sets and their fields using @id
print("Available Record Sets and Their Fields:\n")
record_sets = list(dataset.metadata.record_sets)
for i, record_set in enumerate(record_sets):
    print(f"{i+1}. RecordSet Name: {getattr(record_set, 'name', 'N/A')}  |  @id: {record_set.id}")
    fields = getattr(record_set, 'fields', [])
    if fields:
        for field in fields:
            print(f"    - Field Name: {getattr(field, 'name', 'N/A')}, @id: {field.id}, type: {getattr(field, 'data_type', 'N/A')}")
    print()

## 3. Data Extraction
Load data from each record set. RecordSet and Field `@id`s are used to ensure correct and reproducible references.

Below, we extract all records from each available RecordSet for exploration.

In [ ]:
# Create a list of RecordSet @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    # Fetch records as list of dicts, keyed by field @id
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"RecordSet @{record_set_id} loaded: Shape {df.shape}, Columns: {df.columns.tolist()}")
        display(df.head())
    else:
        print(f"RecordSet @{record_set_id} appears empty.")

## 4. Exploratory Data Analysis (EDA)
Apply basic filtering, normalization, and grouping using Field and RecordSet `@id`s.

Below, we select the main survey record set (most likely the first or primary), pick a numeric field (e.g., GAD-7 or PHQ-9 total score), and process as an example.

In [ ]:
# --- Configuration: Set main RecordSet and Field @ids used for EDA ---

# You may need to adjust these if ids or field names change.
# For this dataset, we infer common mental health survey field names; replace with actual @id as needed.
main_record_set_id = record_set_ids[0]  # Example: first RecordSet (main survey data)
df = dataframes[main_record_set_id]

# Print column (field @id) list for reference
print(f"Fields (@id) in main RecordSet @{main_record_set_id}:")
for col in df.columns: print(col)

# Example: Filter and normalize GAD-7 scores if field exists
# Use actual @id from the field list printed above
numeric_field_id = None
for col in df.columns:
    if 'gad7' in col.lower():  # Look for GAD-7 score column
        numeric_field_id = col
        break
if numeric_field_id is None:
    # Fallback: use first numerical-looking column
    for col in df.select_dtypes(include=np.number).columns:
        numeric_field_id = col
        break
if numeric_field_id is None:
    raise RuntimeError("No numeric field found for EDA.")

threshold = 5  # Example threshold for GAD-7 moderate anxiety
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records where {numeric_field_id} > {threshold} (n={len(filtered_df)}):")
display(filtered_df.head())

# Normalize the scores
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()

print(f"First few normalized scores for {numeric_field_id}:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by categorical field if available (e.g., gender field @id)
group_field_id = None
for col in df.columns:
    if 'gender' in col.lower() or 'sex' in col.lower():
        group_field_id = col
        break
if group_field_id and group_field_id in filtered_df.columns:
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"Mean {numeric_field_id} by {group_field_id}:")
    print(grouped)
else:
    print('No suitable group field (@id like gender/sex) found for grouping.')

## 5. Visualization
Visualize the distribution of the selected numeric field and group-level means where available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of (filtered) GAD-7 or chosen score
plt.figure(figsize=(8, 4))
sns.histplot(filtered_df[numeric_field_id], bins=10, kde=True)
plt.title(f"Distribution of {numeric_field_id} (> {threshold})")
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

# Plot group means if grouping field exists
if group_field_id and group_field_id in filtered_df.columns:
    plt.figure(figsize=(6, 4))
    means = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    sns.barplot(x=means.index.astype(str), y=means.values)
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.show()

## 6. Conclusion

In this notebook, we explored the FAIR² mental health survey dataset structured with the Croissant schema. Using only `@id` identifiers for each entity ensures reproducibility and consistency. We loaded all record sets, examined fields, filtered for higher GAD-7 scores, normalized scores, and visualized distributions and group differences. This AI-ready dataset enables further analysis of mental health indicators in Kilifi County and can be adapted by referencing exact record set and field `@id`s.

**Next Steps:**
- Extend EDA to include other psychometric scores (e.g. PHQ-9, PSQ) by their field `@id`s.
- Apply advanced machine learning models using the normalized data.
- Explore survey completion trends or conduct subgroup analysis by additional demographics, always referencing `@id` fields for rigor.